# Creating Database "LogisticsAnalyticsDB"

In [1]:
import pyodbc
import pandas as pd

In [2]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};" # Use print(pyodbc.drivers())
    "SERVER=DESKTOP-0SIEUH6\SQLEXPRESS;"
    "DATABASE=master;"
    "Trusted_Connection=yes;",
    autocommit=True)

cursor = conn.cursor()

cursor.execute("create database LogisticsAnalyticsDB;")

cursor.close()

conn.close()

# Using "LogisticsAnalyticsDB"

In [3]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-0SIEUH6\\SQLEXPRESS;"
    "DATABASE=LogisticsAnalyticsDB;"   # <-- note: LogisticsAnalyticsDB, not master
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

# Table Dim_Dates

In [4]:
Dim_Dates = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Dates.csv")

print(Dim_Dates.dtypes)
print(Dim_Dates.head())

DateKey       int64
FullDate     object
Day           int64
Month         int64
MonthName    object
Quarter       int64
Year          int64
DayOfWeek    object
IsWeekend      bool
IsHoliday      bool
dtype: object
    DateKey    FullDate  Day  Month MonthName  Quarter  Year  DayOfWeek  \
0  20220101  2022-01-01    1      1   January        1  2022   Saturday   
1  20220102  2022-01-02    2      1   January        1  2022     Sunday   
2  20220103  2022-01-03    3      1   January        1  2022     Monday   
3  20220104  2022-01-04    4      1   January        1  2022    Tuesday   
4  20220105  2022-01-05    5      1   January        1  2022  Wednesday   

   IsWeekend  IsHoliday  
0       True       True  
1       True      False  
2      False      False  
3      False      False  
4      False      False  


In [5]:
cursor.execute("""
               CREATE TABLE dbo.Dim_Dates (
    DateKey         INT             NOT NULL PRIMARY KEY,   -- yyyymmdd
    FullDate        DATE            NOT NULL,
    [Day]           TINYINT         NOT NULL,
    [Month]         TINYINT         NOT NULL,
    MonthName       VARCHAR(15)     NOT NULL,
    Quarter         TINYINT         NOT NULL,
    [Year]          SMALLINT        NOT NULL,
    DayOfWeek       VARCHAR(10)     NOT NULL,
    IsWeekend       BIT             NOT NULL,
    IsHoliday       BIT             NOT NULL
);
               """)
conn.commit()

cursor.fast_executemany = True   # much faster for bulk inserts

# Convert DataFrame rows into a list of tuples matching your table's column order
Dim_Dates = list(Dim_Dates.itertuples(index=False, name=None))

cursor.executemany("""
    INSERT INTO dbo.Dim_Dates 
    (DateKey, FullDate, [Day], [Month], MonthName, Quarter, [Year], DayOfWeek, IsWeekend, IsHoliday)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", Dim_Dates)

conn.commit()

print(f"Inserted {len(Dim_Dates)} rows into Dim_Dates.")

cursor.execute("""select top 3 * from Dim_Dates""")
cursor.fetchall()

Inserted 1096 rows into Dim_Dates.


[(20220101, datetime.date(2022, 1, 1), 1, 1, 'January', 1, 2022, 'Saturday', True, True),
 (20220102, datetime.date(2022, 1, 2), 2, 1, 'January', 1, 2022, 'Sunday', True, False),
 (20220103, datetime.date(2022, 1, 3), 3, 1, 'January', 1, 2022, 'Monday', False, False)]

# Table Dim_Carriers

In [6]:
Dim_Carriers = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Carriers.csv")

print(Dim_Carriers.info(), "\n\n\n")
print(Dim_Carriers)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CarrierID           10 non-null     int64  
 1   CarrierName         10 non-null     object 
 2   CarrierType         10 non-null     object 
 3   ServiceLevel        10 non-null     object 
 4   HistoricOnTimeRate  10 non-null     float64
dtypes: float64(1), int64(1), object(3)
memory usage: 532.0+ bytes
None 



   CarrierID          CarrierName  CarrierType ServiceLevel  \
0          1     BlueDart Express  Third-Party     Standard   
1          2          Swift Cargo     In-House      Express   
2          3  FastTrack Logistics  Third-Party     Standard   
3          4    DeltaLine Freight  Third-Party      Premium   
4          5   Continental Movers     In-House     Standard   
5          6    Skyline Transport  Third-Party      Express   
6          7       Rapid Roadways  Thir

In [7]:
# Creating table

cursor.execute("""CREATE TABLE dbo.Dim_Carriers (
    CarrierID          INT            NOT NULL PRIMARY KEY,
    CarrierName        VARCHAR(50)    NOT NULL,
    CarrierType        VARCHAR(20)    NOT NULL,     -- Third-Party / In-House
    ServiceLevel       VARCHAR(20)    NOT NULL,     -- Standard / Express / Premium
    HistoricOnTimeRate DECIMAL(5,4)   NOT NULL       -- NOT to be used directly as an ML feature for new shipments (see data dictionary note)
);""")


conn.commit()

# inserting data into table

Dim_Carriers = list(Dim_Carriers.itertuples(index=False, name=None))


cursor.executemany("""
    INSERT INTO dbo.Dim_Carriers
    (CarrierID, CarrierName, CarrierType, ServiceLevel, HistoricOnTimeRate)
    Values (?, ?, ?, ?, ?)""", Dim_Carriers)

conn.commit()
print(f"Inserted {len(Dim_Carriers)} rows into Dim_Carriers.")

cursor.execute("select top 3 * from Dim_Carriers")
cursor.fetchall()

Inserted 10 rows into Dim_Carriers.


[(1, 'BlueDart Express', 'Third-Party', 'Standard', Decimal('0.9000')),
 (2, 'Swift Cargo', 'In-House', 'Express', Decimal('0.7800')),
 (3, 'FastTrack Logistics', 'Third-Party', 'Standard', Decimal('0.8200'))]

# Table Dim_Locations

In [8]:
Dim_Locations = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Locations.csv")

print(Dim_Locations.info())
print(Dim_Locations.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   LocationID  25 non-null     int64 
 1   City        25 non-null     object
 2   State       25 non-null     object
 3   Region      25 non-null     object
 4   Country     25 non-null     object
 5   Zone        25 non-null     object
dtypes: int64(1), object(5)
memory usage: 1.3+ KB
None
   LocationID       City        State Region Country    Zone
0           1     Mumbai  Maharashtra   West   India  Zone-C
1           2      Delhi        Delhi  North   India  Zone-A
2           3  Bengaluru    Karnataka  South   India  Zone-A
3           4    Chennai   Tamil Nadu  South   India  Zone-C
4           5    Kolkata  West Bengal   East   India  Zone-B


In [9]:
cursor.execute("""
            CREATE TABLE dbo.Dim_Locations (
    LocationID      INT             NOT NULL PRIMARY KEY,
    City            VARCHAR(50)     NOT NULL,
    [State]         VARCHAR(50)     NOT NULL,
    Region          VARCHAR(20)     NOT NULL,
    Country         VARCHAR(30)     NOT NULL,
    Zone            VARCHAR(10)     NOT NULL
);
            """)

conn.commit()

Dim_Locations = list(Dim_Locations.itertuples(index=False, name=None))

cursor.executemany("""Insert into  dbo.Dim_Locations
                  (LocationID, City, [State], Region, Country, Zone)
                  VALUES (?, ?, ?, ?, ?, ?);""", Dim_Locations)

conn.commit()

cursor.execute("select top 3 * from Dim_Locations")
cursor.fetchall()

[(1, 'Mumbai', 'Maharashtra', 'West', 'India', 'Zone-C'),
 (2, 'Delhi', 'Delhi', 'North', 'India', 'Zone-A'),
 (3, 'Bengaluru', 'Karnataka', 'South', 'India', 'Zone-A')]

# Table Dim_Customers

In [10]:
Dim_Customers = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Customers.csv")

Dim_Customers

print(Dim_Customers.info(), "\n\n\n")
print(Dim_Customers.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   CustomerID        3000 non-null   int64 
 1   CustomerName      3000 non-null   object
 2   Segment           3000 non-null   object
 3   LocationID        3000 non-null   int64 
 4   RegistrationDate  3000 non-null   object
 5   CustomerTier      3000 non-null   object
dtypes: int64(2), object(4)
memory usage: 140.8+ KB
None 



   CustomerID    CustomerName     Segment  LocationID RegistrationDate  \
0           1  Customer_00001  Enterprise          16       2020-03-04   
1           2  Customer_00002      Retail          21       2021-12-04   
2           3  Customer_00003         SME          17       2019-03-29   

  CustomerTier  
0       Silver  
1       Bronze  
2       Silver  


In [11]:
cursor.execute("""
                CREATE TABLE dbo.Dim_Customers (
    CustomerID       INT            NOT NULL PRIMARY KEY,
    CustomerName     VARCHAR(50)    NOT NULL,
    Segment          VARCHAR(20)    NOT NULL,      -- Retail/Wholesale/Enterprise/SME (some dirty casing by design)
    LocationID       INT            NOT NULL,
    RegistrationDate DATE           NOT NULL,
    CustomerTier     VARCHAR(15)    NOT NULL,      -- Bronze/Silver/Gold/Platinum
    CONSTRAINT FK_Customers_Locations FOREIGN KEY (LocationID) REFERENCES dbo.Dim_Locations(LocationID)
);
""")

conn.commit()

Dim_Customers = list(Dim_Customers.itertuples(index=False, name=None))

cursor.executemany("""insert into dbo.Dim_Customers
                    (CustomerID, CustomerName, Segment, LocationID, RegistrationDate, CustomerTier)
                    values(?, ?, ?, ?, ?, ?);
                    """, Dim_Customers)

conn.commit()

cursor.execute("select top 3 * from Dim_Customers;")
cursor.fetchall()

[(1, 'Customer_00001', 'Enterprise', 16, datetime.date(2020, 3, 4), 'Silver'),
 (2, 'Customer_00002', 'Retail', 21, datetime.date(2021, 12, 4), 'Bronze'),
 (3, 'Customer_00003', 'SME', 17, datetime.date(2019, 3, 29), 'Silver')]

# Table Dim_Warehouses

In [12]:
Dim_Warehouses = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Warehouses.csv")

print(Dim_Warehouses.info(), "\n\n\n")
print(Dim_Warehouses.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   WarehouseID      30 non-null     int64 
 1   WarehouseName    30 non-null     object
 2   LocationID       30 non-null     int64 
 3   WarehouseType    30 non-null     object
 4   CapacityUnits    30 non-null     int64 
 5   AutomationLevel  30 non-null     object
dtypes: int64(3), object(3)
memory usage: 1.5+ KB
None 



   WarehouseID  WarehouseName  LocationID       WarehouseType  CapacityUnits  \
0            1  Warehouse_001           2        Regional Hub          17715   
1            2  Warehouse_002          13  Fulfillment Center          34039   
2            3  Warehouse_003          14        Regional Hub          44670   

  AutomationLevel  
0          Medium  
1            High  
2            High  


In [13]:
cursor.execute("""
                CREATE TABLE dbo.Dim_Warehouses (
    WarehouseID      INT            NOT NULL PRIMARY KEY,
    WarehouseName    VARCHAR(50)    NOT NULL,
    LocationID       INT            NOT NULL,
    WarehouseType    VARCHAR(25)    NOT NULL,
    CapacityUnits    INT            NOT NULL,
    AutomationLevel  VARCHAR(10)    NOT NULL,       -- Low/Medium/High
    CONSTRAINT FK_Warehouses_Locations FOREIGN KEY (LocationID) REFERENCES dbo.Dim_Locations(LocationID)
);
                """)

conn.commit()

Dim_Warehouses = list(Dim_Warehouses.itertuples(index=False, name=None))

cursor.executemany("""
                    insert into dbo.Dim_Warehouses
                    (WarehouseID, WarehouseName, LocationID, WarehouseType, CapacityUnits, AutomationLevel)
                    values (?,?,?,?,?,?)""",
    Dim_Warehouses)

conn.commit()

cursor.execute("select top 3 * from Dim_Warehouses;")
cursor.fetchall()

[(1, 'Warehouse_001', 2, 'Regional Hub', 17715, 'Medium'),
 (2, 'Warehouse_002', 13, 'Fulfillment Center', 34039, 'High'),
 (3, 'Warehouse_003', 14, 'Regional Hub', 44670, 'High')]

# Table Dim_Employees

In [14]:
Dim_Employees = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Employees.csv")

print(Dim_Employees.info(), "\n\n\n")
print(Dim_Employees.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   EmployeeID       900 non-null    int64  
 1   WarehouseID      900 non-null    int64  
 2   Role             900 non-null    object 
 3   ExperienceYears  900 non-null    float64
 4   ShiftType        900 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 35.3+ KB
None 



   EmployeeID  WarehouseID                Role  ExperienceYears ShiftType
0           1           19          Supervisor              2.8     Night
1           2           12              Loader              5.3       Day
2           3            4  Driver Coordinator              4.5     Night


In [15]:
cursor.execute("""
                CREATE TABLE dbo.Dim_Employees (
    EmployeeID       INT            NOT NULL PRIMARY KEY,
    WarehouseID      INT            NOT NULL,
    Role             VARCHAR(30)    NOT NULL,
    ExperienceYears  DECIMAL(4,1)   NOT NULL,
    ShiftType        VARCHAR(15)    NOT NULL,       -- Day/Night/Rotational
    CONSTRAINT FK_Employees_Warehouses FOREIGN KEY (WarehouseID) REFERENCES dbo.Dim_Warehouses(WarehouseID)
);
                """)

conn.commit()

Dim_Employees = list(Dim_Employees.itertuples(index=False, name=None))

cursor.executemany("""
                    insert into dbo.Dim_Employees
                    (EmployeeID, WarehouseID, Role, ExperienceYears, ShiftType)
                    values (?, ?, ?, ?, ?)
                    """,
                    Dim_Employees)
conn.commit()

cursor.execute("select top 3 * from Dim_Employees;")
cursor.fetchall()

[(1, 19, 'Supervisor', Decimal('2.8'), 'Night'),
 (2, 12, 'Loader', Decimal('5.3'), 'Day'),
 (3, 4, 'Driver Coordinator', Decimal('4.5'), 'Night')]

# Table Dim_Products

In [16]:
Dim_Products = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Products.csv")

print(Dim_Products.info(), "\n\n\n")
print(Dim_Products.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ProductID     800 non-null    int64  
 1   ProductName   800 non-null    object 
 2   Category      800 non-null    object 
 3   SubCategory   800 non-null    object 
 4   UnitWeightKg  800 non-null    float64
 5   UnitVolumeM3  800 non-null    float64
 6   FragileFlag   800 non-null    bool   
dtypes: bool(1), float64(2), int64(1), object(3)
memory usage: 38.4+ KB
None 



   ProductID    ProductName     Category         SubCategory  UnitWeightKg  \
0          1  Product_00001    Furniture    Office Furniture          7.67   
1          2  Product_00002      Grocery  Household Supplies          3.69   
2          3  Product_00003  Electronics     Home Appliances          2.16   

   UnitVolumeM3  FragileFlag  
0        0.0146         True  
1        0.0058        False  
2        0.0592       

In [17]:
cursor.execute("""
                CREATE TABLE dbo.Dim_Products (
    ProductID        INT            NOT NULL PRIMARY KEY,
    ProductName      VARCHAR(50)    NOT NULL,
    Category         VARCHAR(30)    NOT NULL,
    SubCategory      VARCHAR(40)    NOT NULL,
    UnitWeightKg     DECIMAL(8,2)   NOT NULL,
    UnitVolumeM3     DECIMAL(8,4)   NOT NULL,
    FragileFlag      BIT            NOT NULL
);
                """)

conn.commit()

Dim_Products = list(Dim_Products.itertuples(index=False, name=None))

cursor.executemany("""
                insert into dbo.Dim_Products
                (ProductID, ProductName, Category, SubCategory, UnitWeightKg, UnitVolumeM3, FragileFlag)
                values (?, ?, ?, ?, ?, ?, ?)""", Dim_Products)

conn.commit()

cursor.execute("select top 3 * from Dim_Products")
cursor.fetchall()

[(1, 'Product_00001', 'Furniture', 'Office Furniture', Decimal('7.67'), Decimal('0.0146'), True),
 (2, 'Product_00002', 'Grocery', 'Household Supplies', Decimal('3.69'), Decimal('0.0058'), False),
 (3, 'Product_00003', 'Electronics', 'Home Appliances', Decimal('2.16'), Decimal('0.0592'), False)]

# Table Dim_Routes

In [18]:
Dim_Routes = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Routes.csv")

print(Dim_Routes.info(), "\n\n\n")
print(Dim_Routes.tail(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RouteID                350 non-null    int64  
 1   OriginLocationID       350 non-null    int64  
 2   DestinationLocationID  350 non-null    int64  
 3   RouteType              350 non-null    object 
 4   DistanceKm             350 non-null    float64
 5   AvgTrafficLevel        350 non-null    object 
dtypes: float64(1), int64(3), object(2)
memory usage: 16.5+ KB
None 



     RouteID  OriginLocationID  DestinationLocationID           RouteType  \
347      348                 9                     19  National Long-Haul   
348      349                18                      2  National Long-Haul   
349      350                17                      7  National Long-Haul   

     DistanceKm AvgTrafficLevel  
347      1466.4             Low  
348      1603.9            High

In [19]:
cursor.execute("""
                CREATE TABLE dbo.Dim_Routes (
    RouteID              INT            NOT NULL PRIMARY KEY,
    OriginLocationID     INT            NOT NULL,
    DestinationLocationID INT           NOT NULL,
    RouteType            VARCHAR(25)    NOT NULL,   -- Intra-City/Intra-State/Inter-State/National Long-Haul
    DistanceKm           DECIMAL(8,1)   NOT NULL,
    AvgTrafficLevel      VARCHAR(10)    NOT NULL,   -- Low/Medium/High
    CONSTRAINT FK_Routes_Origin FOREIGN KEY (OriginLocationID) REFERENCES dbo.Dim_Locations(LocationID),
    CONSTRAINT FK_Routes_Dest   FOREIGN KEY (DestinationLocationID) REFERENCES dbo.Dim_Locations(LocationID)
);
                """)

conn.commit()

Dim_Routes = list(Dim_Routes.itertuples(index=False, name = None))

cursor.executemany("""
                insert into dbo.Dim_Routes
                (RouteID, OriginLocationID, DestinationLocationID, RouteType, DistanceKm, AvgTrafficLevel)
                values(?, ?, ?, ?, ?, ?)
                """, Dim_Routes)

conn.commit()

cursor.execute("select top 3 * from Dim_Routes;")
cursor.fetchall()

[(1, 20, 4, 'National Long-Haul', Decimal('1081.5'), 'Medium'),
 (2, 19, 24, 'National Long-Haul', Decimal('2065.3'), 'High'),
 (3, 17, 22, 'Inter-State', Decimal('1339.3'), 'Low')]

# Table Dim_Vehicles

In [20]:
Dim_Vehicles = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Dim_Vehicles.csv")

print(Dim_Vehicles.info())
print(Dim_Vehicles.tail(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   VehicleID        500 non-null    int64  
 1   CarrierID        500 non-null    int64  
 2   VehicleType      500 non-null    object 
 3   CapacityKg       500 non-null    float64
 4   FuelType         500 non-null    object 
 5   VehicleAgeYears  500 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 23.6+ KB
None
     VehicleID  CarrierID VehicleType  CapacityKg  FuelType  VehicleAgeYears
497        498          3  Mini Truck      2768.8  Electric                6
498        499          9         Van       319.8    Diesel                6
499        500          3         Van      2010.9       CNG               11


In [21]:
cursor.execute("""
                CREATE TABLE dbo.Dim_Vehicles (
    VehicleID        INT            NOT NULL PRIMARY KEY,
    CarrierID        INT            NOT NULL,
    VehicleType      VARCHAR(25)    NOT NULL,
    CapacityKg       DECIMAL(9,1)   NOT NULL,
    FuelType         VARCHAR(15)    NOT NULL,
    VehicleAgeYears  TINYINT        NOT NULL,
    CONSTRAINT FK_Vehicles_Carriers FOREIGN KEY (CarrierID) REFERENCES dbo.Dim_Carriers(CarrierID)
);
                """)


conn.commit()

Dim_Vehicles = list(Dim_Vehicles.itertuples(index=False, name=None))

cursor.executemany("""
                    insert into dbo.Dim_Vehicles
                    (VehicleID, CarrierID, VehicleType, CapacityKg, FuelType, VehicleAgeYears)
                    values(?, ?, ?, ?, ?, ?)
                    """, Dim_Vehicles)

conn.commit()

cursor.execute("select top 3 * from Dim_Vehicles;")
cursor.fetchall()

[(1, 5, 'Container Truck', Decimal('1513.8'), 'CNG', 14),
 (2, 9, 'Two-Wheeler', Decimal('1618.4'), 'Petrol', 8),
 (3, 2, 'Mini Truck', Decimal('1745.7'), 'Diesel', 13)]

# Table Fact_Shipments

In [24]:
Fact_Shipments = pd.read_csv(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\datasets\Fact_Shipments.csv")

print(Fact_Shipments.info())
print(Fact_Shipments.tail(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22110 entries, 0 to 22109
Data columns (total 30 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ShipmentID                    22110 non-null  int64  
 1   DateKey                       22110 non-null  int64  
 2   CustomerID                    22110 non-null  int64  
 3   ProductID                     22110 non-null  int64  
 4   CarrierID                     22110 non-null  int64  
 5   VehicleID                     22110 non-null  int64  
 6   WarehouseID                   22110 non-null  int64  
 7   EmployeeID                    21888 non-null  float64
 8   RouteID                       22110 non-null  int64  
 9   ShipmentStatus                22110 non-null  object 
 10  DeliveryMode                  22110 non-null  object 
 11  ShippingPriority              22110 non-null  object 
 12  PaymentMethod                 22110 non-null  object 
 13  W

In [25]:
# As this table contains nulls
# Convert pandas NaN -> Python None before inserting, since pyodbc can't send NaN as a SQL parameter


Fact_Shipments = Fact_Shipments.astype(object).where(pd.notnull(Fact_Shipments), None)

In [26]:
cursor.execute("""
                CREATE TABLE dbo.Fact_Shipments (
    ShipmentID                     INT            NOT NULL PRIMARY KEY,
    DateKey                        INT            NOT NULL,
    CustomerID                     INT            NOT NULL,
    ProductID                      INT            NOT NULL,
    CarrierID                      INT            NOT NULL,
    VehicleID                      INT            NOT NULL,
    WarehouseID                    INT            NOT NULL,
    EmployeeID                     INT            NULL,      -- nullable: some records missing by design
    RouteID                        INT            NOT NULL,
    ShipmentStatus                 VARCHAR(15)    NOT NULL,  -- Delivered/In Transit/Returned/Cancelled
    DeliveryMode                   VARCHAR(10)    NOT NULL,  -- Road/Rail/Air/Sea (some dirty lowercase by design)
    ShippingPriority               VARCHAR(15)    NOT NULL,  -- Standard/Express/Same-Day
    PaymentMethod                  VARCHAR(15)    NOT NULL,  -- Prepaid/COD/Credit Terms
    WeatherCondition                VARCHAR(15)   NULL,      -- nullable: some records missing by design
    TrafficCondition               VARCHAR(10)    NOT NULL,  -- Low/Medium/High (some dirty casing by design)
    OrderQuantity                  INT            NOT NULL,
    PackageWeightKg                DECIMAL(10,2)  NULL,      -- nullable + contains outliers by design
    PackageVolumeM3                DECIMAL(10,4)  NULL,
    DistanceKm                     DECIMAL(10,1)  NOT NULL,  -- contains a few negative/invalid values by design
    ShippingCost                   DECIMAL(10,2)  NULL,      -- contains outliers by design
    OrderValue                     DECIMAL(12,2)  NOT NULL,
    HandlingTimeHours              DECIMAL(6,2)   NOT NULL,
    LoadingTimeHours               DECIMAL(6,2)   NOT NULL,
    WarehouseProcessingTimeHours   DECIMAL(6,2)   NULL,
    FuelConsumptionLiters          DECIMAL(8,2)   NULL,
    PromisedDeliveryDays           TINYINT        NOT NULL,  -- known at booking time — SAFE to use as ML feature
    ActualDeliveryDays             TINYINT        NOT NULL,  -- known only AFTER delivery — DO NOT use as ML feature (leakage)
    DeliveryTimeHours              DECIMAL(8,1)   NOT NULL,  -- known only AFTER delivery — DO NOT use as ML feature (leakage)
    NumPreviousDelaysCustomer      TINYINT        NOT NULL,
    Late_Delivery                  BIT            NOT NULL,  -- TARGET VARIABLE

    CONSTRAINT FK_Fact_Dates       FOREIGN KEY (DateKey)     REFERENCES dbo.Dim_Dates(DateKey),
    CONSTRAINT FK_Fact_Customers   FOREIGN KEY (CustomerID)  REFERENCES dbo.Dim_Customers(CustomerID),
    CONSTRAINT FK_Fact_Products    FOREIGN KEY (ProductID)   REFERENCES dbo.Dim_Products(ProductID),
    CONSTRAINT FK_Fact_Carriers    FOREIGN KEY (CarrierID)   REFERENCES dbo.Dim_Carriers(CarrierID),
    CONSTRAINT FK_Fact_Vehicles    FOREIGN KEY (VehicleID)   REFERENCES dbo.Dim_Vehicles(VehicleID),
    CONSTRAINT FK_Fact_Warehouses  FOREIGN KEY (WarehouseID) REFERENCES dbo.Dim_Warehouses(WarehouseID),
    CONSTRAINT FK_Fact_Employees   FOREIGN KEY (EmployeeID)  REFERENCES dbo.Dim_Employees(EmployeeID),
    CONSTRAINT FK_Fact_Routes      FOREIGN KEY (RouteID)     REFERENCES dbo.Dim_Routes(RouteID),

    CONSTRAINT CK_Fact_Late_Delivery CHECK (Late_Delivery IN (0,1)),
    CONSTRAINT CK_Fact_OrderQuantity CHECK (OrderQuantity > 0)
);
                """)

conn.commit()

Fact_Shipments = list(Fact_Shipments.itertuples(index=False, name=None))

cursor.executemany("""
                    insert into dbo.Fact_Shipments
                    (ShipmentID, DateKey, CustomerID, ProductID, CarrierID, VehicleID, WarehouseID, EmployeeID, 
                    RouteID, ShipmentStatus, DeliveryMode, ShippingPriority, PaymentMethod, WeatherCondition, TrafficCondition, 
                    OrderQuantity, PackageWeightKg, PackageVolumeM3, DistanceKm, ShippingCost, OrderValue, HandlingTimeHours, 
                    LoadingTimeHours, WarehouseProcessingTimeHours, FuelConsumptionLiters, PromisedDeliveryDays, 
                    ActualDeliveryDays, DeliveryTimeHours, NumPreviousDelaysCustomer, Late_Delivery)
                    values (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    """, Fact_Shipments)

conn.commit()

cursor.execute("select top 3 * from Fact_Shipments;")
cursor.fetchall()

[(1, 20240816, 1690, 456, 9, 315, 7, 816, 27, 'Delivered', 'Road', 'Express', 'COD', 'Clear', 'High', 21, Decimal('100.70'), Decimal('0.6140'), Decimal('826.9'), Decimal('1148.45'), Decimal('6795.59'), Decimal('0.11'), Decimal('0.83'), Decimal('1.96'), Decimal('167.59'), 2, 1, Decimal('24.2'), 1, False),
 (2, 20220209, 2755, 785, 2, 74, 23, 549, 303, 'Delivered', 'Sea', 'Express', 'Prepaid', 'Clear', 'Low', 3, Decimal('6.74'), Decimal('0.1811'), Decimal('1410.5'), Decimal('2366.69'), Decimal('6048.13'), Decimal('1.42'), Decimal('0.87'), Decimal('5.08'), Decimal('156.02'), 3, 2, Decimal('48.0'), 2, False),
 (3, 20221024, 2601, 376, 10, 56, 22, 676, 147, 'Returned', 'Road', 'Standard', 'COD', 'Clear', 'HIGH ', 5, Decimal('1.10'), Decimal('0.3300'), Decimal('1243.0'), Decimal('1825.66'), Decimal('2420.39'), Decimal('1.04'), Decimal('0.90'), Decimal('2.88'), Decimal('189.48'), 5, 4, Decimal('98.3'), 1, False)]

# ---------------------------- Database Created sucessfully ----------------------------